# ORIE 5132 — Choice Modeling, Assortment Optimization, and PricingThis notebook contains Problems 1 through 5. Problems 6 and 7 are in a separate notebook.- **Problem 1**: MNL parameter estimation by MLE on the full dataset- **Problem 2**: Assortment optimization under MNL for the four small datasets- **Problem 3**: Pricing under MNL for the four small datasets- **Problem 4**: Mixture of MNL with early vs. late customer types- **Problem 5**: Assortment optimization under the mixture MNL

## Setup

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.special import logsumexp

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
DATA_DIR = "/Users/jamesb/Code/Pricing/Pricing_Project/data/"

FEATURES = [
    "prop_starrating",
    "prop_review_score",
    "prop_brand_bool",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd",
    "promotion_flag",
]

PARAM_NAMES = ["intercept"] + FEATURES

### Load and validate the data

In [ ]:
df = pd.read_csv(DATA_DIR + "data.csv")

required = {"srch_id", "srch_booking_window", "booking_bool", *FEATURES}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"data.csv is missing required columns: {missing}")

df = df.dropna(subset=list(required)).copy()

print(f"Rows: {len(df):,}")
print(f"Unique search queries: {df['srch_id'].nunique():,}")
df.head(10)

## Problem 1 — MNL ModelCustomers are assumed to choose according to an MNL model. For hotel $j$ with features $x_{ji}$ and customer feature sensitivities $\beta_i$:$$u_j = \beta_0 + \sum_{i=1}^8 \beta_i x_{ji}, \quad v_j = e^{u_j}, \quad \mathbb{P}(j \mid S) = \frac{v_j}{1 + \sum_{p \in S} v_p}$$The log-likelihood for one search query is$$\ell_q(\beta) = \sum_{j \in S_q} y_j u_j - \log\!\left(1 + \sum_{j \in S_q} e^{u_j}\right)$$where $y_j$ is the booking indicator. We estimate $\beta_0,\dots,\beta_8$ by maximum likelihood.

In [ ]:
X = df[FEATURES].to_numpy(dtype=float)
y = df["booking_bool"].to_numpy(dtype=float)
groups = df["srch_id"].to_numpy()

In [ ]:
def compute_utilities(beta, X):
    return beta[0] + X @ beta[1:]


def prepare_groups(groups):
    """Map raw srch_id values to dense indices 0..n_groups-1 for fast bincount."""
    _, group_index = np.unique(groups, return_inverse=True)
    return group_index


def neg_log_likelihood(beta, X, y, group_index):
    u = compute_utilities(beta, X)

    booked_utility = y @ u

    # vectorized: log(1 + sum exp(u_j)) summed over groups
    sum_exp_by_group = np.bincount(group_index, weights=np.exp(u))
    log_denom = np.log1p(sum_exp_by_group).sum()

    return -(booked_utility - log_denom)

In [ ]:
group_index = prepare_groups(groups)
beta_init = np.zeros(len(PARAM_NAMES))

result = minimize(
    neg_log_likelihood,
    beta_init,
    args=(X, y, group_index),
    method="L-BFGS-B",
    options={"maxiter": 1000},
)

beta_hat = result.x

print(f"Converged: {result.success}; negative log-likelihood: {result.fun:.4f}")
print()
for name, val in zip(PARAM_NAMES, beta_hat):
    print(f"{name:>30s}: {val:+.4f}")

### Commentary on coefficientsAll signs are intuitively correct:- **prop_starrating** (+0.48) and **promotion_flag** (+0.46) are the strongest positive drivers — higher star rating and promotions both boost booking probability.- **prop_brand_bool** (+0.23) and **prop_review_score** (+0.12) are also positive, consistent with customers preferring branded hotels and higher reviews.- **prop_accesibility_score** (+0.57) is surprisingly the largest positive coefficient. It is likely capturing something correlated with desirability beyond pure accessibility.- **prop_location_score** (+0.02) is nearly zero, possibly due to limited variance in the column or collinearity with other features.- **price_usd** (-0.0073) is correctly negative. Per dollar this looks small, but prices range $40–$600, so a $100 increase reduces utility by ~0.73 — comparable to losing 1.5 star rating points.- **prop_log_historical_price** (-0.035) is also negative; its correlation with `price_usd` likely splits the price-sensitivity effect across the two coefficients.

## Problem 2 — Assortment Optimization under MNLFor each of `data1.csv`, `data2.csv`, `data3.csv`, `data4.csv`, find the optimal subset $S$ of hotels to display:$$R(S) = \frac{\sum_{j \in S} p_j v_j}{1 + \sum_{j \in S} v_j}$$By the **revenue-ordered property** of MNL, the optimal $S$ is always a prefix of the price-sorted list. Walk down the sorted list and add hotel $k$ if and only if $p_k > R(\text{current})$. Once a hotel fails this test, all remaining hotels also fail (they have lower prices), so we can stop.

In [ ]:
def optimal_assortment(df_hotels, beta, features=FEATURES):
    df_local = df_hotels.copy()

    u = compute_utilities(beta, df_local[features].to_numpy(dtype=float))
    df_local["v"] = np.exp(u)

    df_local = df_local.sort_values("price_usd", ascending=False).reset_index(drop=True)

    num = 0.0  # sum of p_j * v_j
    den = 1.0  # 1 + sum of v_j (the 1 represents the no-purchase option)
    chosen_idx = []

    for i, row in df_local.iterrows():
        p = row["price_usd"]
        v = row["v"]
        R_current = num / den

        if p > R_current:
            num += p * v
            den += v
            chosen_idx.append(i)
        else:
            break

    S = df_local.loc[chosen_idx]
    R_opt = num / den
    return S, R_opt

In [ ]:
for i in range(1, 5):
    df_i = pd.read_csv(f"{DATA_DIR}data{i}.csv")
    S, R_opt = optimal_assortment(df_i, beta_hat)
    print(f"--- data{i}.csv ---")
    print(f"Optimal assortment size: {len(S)} hotels")
    print(f"Expected revenue: ${R_opt:.2f}")
    print(S[["price_usd", "v"]].to_string())
    print()

### DiscussionAcross the four datasets the optimal assortments range from 10 to 18 hotels and expected revenues from \$97 to \$131.- **data2** has the highest expected revenue (\$130.99) with only 10 hotels — its hotels combine relatively high prices with healthy preference weights.- **data3** is interesting: a \$603 outlier with $v = 0.003$ is included anyway, because its price is so far above the running expected revenue that it clears the threshold despite a tiny choice probability.- **data4** is weakest at \$97.25 — prices cap at \$195 and the assortment is dominated by hotels around \$100, dragging the average down.

## Problem 3 — Pricing under MNLDisplay all hotels in `data{i}.csv` and choose new prices to maximize expected revenue. Under MNL with a shared price coefficient $\beta_p$, the first-order condition gives$$p_j^* = R^* + \frac{1}{|\beta_p|}, \quad \forall j$$so **all hotels share the same optimal price**. We reduce to a 1D maximization over a scalar $p$:$$R(p) = \frac{p \sum_j e^{a_j + \beta_p p}}{1 + \sum_j e^{a_j + \beta_p p}}$$where $a_j = \beta_0 + \sum_{i \ne \text{price}} \beta_i x_{ji}$ is the intrinsic utility (everything except the price contribution).

In [ ]:
def optimal_pricing(df_hotels, beta, features=FEATURES):
    X_h = df_hotels[features].to_numpy(dtype=float)

    price_idx = features.index("price_usd")
    beta_price = beta[price_idx + 1]   # +1 since beta[0] is the intercept

    # intrinsic utility: zero out the price contribution
    X_no_price = X_h.copy()
    X_no_price[:, price_idx] = 0.0
    a = beta[0] + X_no_price @ beta[1:]

    def neg_revenue(p):
        v = np.exp(a + beta_price * p)
        return -p * v.sum() / (1 + v.sum())

    result = minimize_scalar(neg_revenue, bounds=(1.0, 5000.0), method="bounded")

    p_star = result.x
    R_star = -result.fun
    markup = 1.0 / abs(beta_price)
    return p_star, R_star, markup

In [ ]:
for i in range(1, 5):
    df_i = pd.read_csv(f"{DATA_DIR}data{i}.csv")
    p_star, R_star, markup = optimal_pricing(df_i, beta_hat)
    print(f"--- data{i}.csv ---")
    print(f"  Optimal price (all hotels):    ${p_star:.2f}")
    print(f"  Expected revenue:              ${R_star:.2f}")
    print(f"  Theoretical markup 1/|beta_p|: ${markup:.2f}")
    print(f"  p* - R* (should match markup): ${p_star - R_star:.2f}")
    print()

### DiscussionThe FOC sanity check holds across all four datasets: $p^* - R^* = 1/|\beta_p| = \$136.56$ exactly.- **data2** has the highest optimal price (\$384.87) and revenue (\$248.30) — its hotels have the strongest intrinsic appeal, so customers tolerate the highest pricing.- **data1** and **data3** have similar p* (~\$313) and R* (~\$177), suggesting comparable intrinsic appeal on average even though data3 contained the \$603 outlier in P2.- **data4** lands in between (\$350 / \$214).The optimal prices are much higher than typical market prices in P2's assortments because the firm is now setting prices freely rather than working with the prices in the dataset.

## Problem 4 — Mixture of MNL for Early vs. Late ReservationsCustomer types based on `srch_booking_window` (days):- **Type 1 / early**: `srch_booking_window >= 7`- **Type 2 / late**:  `srch_booking_window < 7`For type $k$:$$u_{jk} = \beta_{0k} + \sum_i \beta_{ik} x_{ji}, \quad v_{jk} = e^{u_{jk}}, \quad \mathbb{P}(j \mid S) = \sum_{k=1}^2 \theta_k \frac{v_{jk}}{1 + \sum_{p \in S} v_{pk}}$$We estimate $\theta_1, \theta_2$ by counting unique queries of each type, and fit one MNL per type by MLE on its rows.

### Define early and late customer typesType shares are computed by **unique search queries** so they are not biased by queries that displayed more hotels.

In [ ]:
def query_type_table(df: pd.DataFrame) -> pd.DataFrame:
    per_query = (
        df[["srch_id", "srch_booking_window"]]
        .drop_duplicates(subset="srch_id")
        .copy()
    )
    per_query["customer_type"] = np.where(
        per_query["srch_booking_window"] >= 7,
        "type_1_early",
        "type_2_late",
    )
    return per_query

per_query = query_type_table(df)
per_query.head(10)

In [ ]:
n_queries = per_query["srch_id"].nunique()
theta_1 = per_query["customer_type"].eq("type_1_early").sum() / n_queries
theta_2 = per_query["customer_type"].eq("type_2_late").sum() / n_queries

theta_table = pd.DataFrame({
    "type": ["type_1_early", "type_2_late"],
    "definition": ["srch_booking_window >= 7", "srch_booking_window < 7"],
    "theta": [theta_1, theta_2],
    "searches": [
        per_query["customer_type"].eq("type_1_early").sum(),
        per_query["customer_type"].eq("type_2_late").sum(),
    ],
})

print(f"theta_1 + theta_2 = {theta_1 + theta_2:.6f}")
theta_table

### Attach customer type to row-level hotel data, then estimate one MNL per type

In [ ]:
df = df.merge(per_query[["srch_id", "customer_type"]], on="srch_id", how="left")

df_early = df[df["customer_type"] == "type_1_early"].copy()
df_late = df[df["customer_type"] == "type_2_late"].copy()

summary_by_type = pd.DataFrame({
    "type": ["type_1_early", "type_2_late"],
    "searches": [df_early["srch_id"].nunique(), df_late["srch_id"].nunique()],
    "rows": [len(df_early), len(df_late)],
    "bookings": [df_early["booking_bool"].sum(), df_late["booking_bool"].sum()],
})
summary_by_type

In [ ]:
def estimate_mnl(df_type, label):
    X_t = df_type[FEATURES].to_numpy(dtype=float)
    y_t = df_type["booking_bool"].to_numpy(dtype=float)
    g_t = prepare_groups(df_type["srch_id"].to_numpy())

    beta_init = np.zeros(len(PARAM_NAMES))

    res = minimize(
        neg_log_likelihood,
        beta_init,
        args=(X_t, y_t, g_t),
        method="L-BFGS-B",
        options={"maxiter": 1000},
    )

    if not res.success:
        print(f"WARNING: {label} did not converge: {res.message}")
    return res.x, res

beta_early, result_early = estimate_mnl(df_early, "type 1 early")
beta_late,  result_late  = estimate_mnl(df_late,  "type 2 late")

print(f"Early: success={result_early.success}; neg-log-lik={result_early.fun:.4f}")
print(f"Late:  success={result_late.success};  neg-log-lik={result_late.fun:.4f}")

In [ ]:
beta_table = pd.DataFrame({
    "parameter": PARAM_NAMES,
    "beta_type_1_early": beta_early,
    "beta_type_2_late": beta_late,
    "early_minus_late": beta_early - beta_late,
    "abs_difference": np.abs(beta_early - beta_late),
})

beta_table.sort_values("abs_difference", ascending=False)

### Commentary- **Mixture weights**: $\theta_1 \approx 0.543$, $\theta_2 \approx 0.457$. Slightly more early bookers than late bookers.- **Largest difference**: `prop_accesibility_score` differs sharply between the two types. Early customers weight accessibility much more heavily.- **Promotions**: late customers respond more strongly to promotion flags and star rating than early ones.- All other coefficients differ by less than 0.1 in absolute terms.

## Problem 5 — Early vs. Late Reservations (Assortment Optimization)For each of `data1.csv`, `data2.csv`, `data3.csv`, `data4.csv`, solve three assortment problems:- $S$: optimal assortment when the arriving customer's type is **unknown** (mixture model).- $S_1$: optimal assortment for a known **type 1** customer.- $S_2$: optimal assortment for a known **type 2** customer.Then compare:- Revenue of $S$ vs. $S_1$ under the type 1 MNL.- Revenue of $S$ vs. $S_2$ under the type 2 MNL.For known types, the revenue-ordered property holds and we use a greedy prefix scan. For the unknown-type mixture problem, the property breaks down so we solve a mixed-integer linear program.

### MILP formulationThe mixture revenue is$$R_{\text{mix}}(x) = \theta_1 \frac{\sum_j x_j r_j v_{j1}}{1 + \sum_j x_j v_{j1}} + \theta_2 \frac{\sum_j x_j r_j v_{j2}}{1 + \sum_j x_j v_{j2}}$$Linearize with the substitutions $z_k = 1 / (1 + \sum_j v_{jk} x_j)$ and $y_{jk} = x_j z_k$. Then the denominator relationship $z_k + \sum_j v_{jk} y_{jk} = 1$ enforces the inverse, and the revenue per type becomes $\sum_j r_j v_{jk} y_{jk}$. Linearize $y_{jk} = x_j z_k$ via:$$y_{jk} \le z_k, \quad y_{jk} \le x_j, \quad y_{jk} \ge z_k - (1 - x_j)$$with $x_j \in \{0,1\}$, $z_k \in [0,1]$, $y_{jk} \in [0,1]$. Maximize $\sum_k \theta_k \sum_j r_j v_{jk} y_{jk}$.

In [ ]:
from ortools.linear_solver import pywraplp

PRICE_COL = "price_usd"


def compute_preference_weights(df_hotels, beta):
    X_h = df_hotels[FEATURES].to_numpy(dtype=float)
    u = compute_utilities(beta, X_h)
    u = np.clip(u, -50, 50)   # prevent overflow in exp
    return np.exp(u)


def expected_revenue_type(x, revenue, v):
    x = np.asarray(x)
    return np.sum(x * revenue * v) / (1 + np.sum(x * v))


def expected_revenue_mixture(x, revenue, v1, v2, theta1, theta2):
    return (
        theta1 * expected_revenue_type(x, revenue, v1)
        + theta2 * expected_revenue_type(x, revenue, v2)
    )

In [ ]:
def optimal_assortment_single_type(df_hotels, v, price_col=PRICE_COL):
    df_local = df_hotels.copy()
    df_local["_original_index"] = np.arange(len(df_local))
    df_local["_v"] = v

    df_local = df_local.sort_values(price_col, ascending=False).reset_index(drop=True)

    num = 0.0
    den = 1.0
    chosen_original_indices = []

    for _, row in df_local.iterrows():
        price = row[price_col]
        weight = row["_v"]
        if price > num / den:
            num += price * weight
            den += weight
            chosen_original_indices.append(int(row["_original_index"]))
        else:
            break

    selected = np.zeros(len(df_hotels), dtype=int)
    selected[chosen_original_indices] = 1
    return selected, num / den

In [ ]:
def solve_assortment_ortools(revenue, v_by_type, theta_by_type, time_limit_seconds=300):
    revenue = np.asarray(revenue, dtype=float)
    v_by_type = [np.asarray(v, dtype=float) for v in v_by_type]

    n = len(revenue)
    K = len(v_by_type)

    solver = pywraplp.Solver.CreateSolver("CBC")
    if solver is None:
        raise RuntimeError("CBC solver is not available. Try reinstalling OR-Tools.")

    solver.SetTimeLimit(int(time_limit_seconds * 1000))

    hotels = range(n)
    types = range(K)

    x = {j: solver.BoolVar(f"x_{j}") for j in hotels}
    z = {k: solver.NumVar(0.0, 1.0, f"z_{k}") for k in types}
    y = {(j, k): solver.NumVar(0.0, 1.0, f"y_{j}_{k}") for j in hotels for k in types}

    for k in types:
        solver.Add(z[k] + solver.Sum(v_by_type[k][j] * y[j, k] for j in hotels) == 1)

    for j in hotels:
        for k in types:
            solver.Add(y[j, k] <= z[k])
            solver.Add(y[j, k] <= x[j])
            solver.Add(y[j, k] >= z[k] - (1 - x[j]))

    objective = solver.Sum(
        theta_by_type[k] * revenue[j] * v_by_type[k][j] * y[j, k]
        for k in types for j in hotels
    )
    solver.Maximize(objective)

    status = solver.Solve()

    if status not in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        print(f"WARNING: solver did not find an optimal/feasible solution. Status: {status}")

    selected = np.array([int(round(x[j].solution_value())) for j in hotels])
    return selected, solver.Objective().Value(), status

In [ ]:
def interpret_status(status):
    if status == pywraplp.Solver.OPTIMAL:    return "OPTIMAL"
    if status == pywraplp.Solver.FEASIBLE:   return "FEASIBLE"
    if status == pywraplp.Solver.INFEASIBLE: return "INFEASIBLE"
    if status == pywraplp.Solver.UNBOUNDED:  return "UNBOUNDED"
    return f"OTHER ({status})"


def solve_problem5_for_dataset(file_path):
    df_hotels = pd.read_csv(file_path).copy()
    revenue = df_hotels[PRICE_COL].to_numpy(dtype=float)

    v1 = compute_preference_weights(df_hotels, beta_early)
    v2 = compute_preference_weights(df_hotels, beta_late)

    # S: unknown customer type — solve MILP
    S, mix_obj, status_mix = solve_assortment_ortools(
        revenue=revenue,
        v_by_type=[v1, v2],
        theta_by_type=[theta_1, theta_2],
    )

    # S1, S2: known type — revenue-ordered greedy is exact
    S1, type1_obj = optimal_assortment_single_type(df_hotels, v1)
    S2, type2_obj = optimal_assortment_single_type(df_hotels, v2)

    rev_S_under_type1  = expected_revenue_type(S,  revenue, v1)
    rev_S1_under_type1 = expected_revenue_type(S1, revenue, v1)
    rev_S_under_type2  = expected_revenue_type(S,  revenue, v2)
    rev_S2_under_type2 = expected_revenue_type(S2, revenue, v2)

    summary = {
        "dataset": file_path.split("/")[-1],
        "n_hotels": len(df_hotels),
        "S_size": int(S.sum()),
        "S1_size": int(S1.sum()),
        "S2_size": int(S2.sum()),
        "mixture_revenue_of_S": mix_obj,
        "revenue_S_under_type1": rev_S_under_type1,
        "revenue_S1_under_type1": rev_S1_under_type1,
        "type1_gain_from_personalization": rev_S1_under_type1 - rev_S_under_type1,
        "revenue_S_under_type2": rev_S_under_type2,
        "revenue_S2_under_type2": rev_S2_under_type2,
        "type2_gain_from_personalization": rev_S2_under_type2 - rev_S_under_type2,
        "status_mix": interpret_status(status_mix),
    }

    selected_hotels = {
        "S_unknown_type": df_hotels.loc[S == 1].copy(),
        "S1_type1":       df_hotels.loc[S1 == 1].copy(),
        "S2_type2":       df_hotels.loc[S2 == 1].copy(),
    }
    return summary, selected_hotels

In [ ]:
dataset_files = [f"{DATA_DIR}data{i}.csv" for i in range(1, 5)]

all_results = []
all_selected_hotels = {}

for file_path in dataset_files:
    print(f"Solving {file_path.split('/')[-1]}...")
    result_summary, selected_hotels = solve_problem5_for_dataset(file_path)
    all_results.append(result_summary)
    all_selected_hotels[file_path] = selected_hotels

results_df = pd.DataFrame(all_results)
results_df

In [ ]:
for file_path in dataset_files:
    name = file_path.split("/")[-1]
    print(f"\n--- {name} ---")
    print("S  (unknown type):", list(all_selected_hotels[file_path]["S_unknown_type"].index))
    print("S1 (type 1):     ", list(all_selected_hotels[file_path]["S1_type1"].index))
    print("S2 (type 2):     ", list(all_selected_hotels[file_path]["S2_type2"].index))

### DiscussionFor each dataset we compared the mixture-optimal assortment $S$ against the type-specific assortments $S_1$ and $S_2$:- Under type 1: $S_1$ performs as well as or slightly better than $S$. Gains are small (on the order of $0$–$\$0.30$).- Under type 2: $S_2$ consistently performs better than $S$ — late customers benefit more from personalization.**Takeaway**: knowing the customer type lets us slightly improve revenue by tailoring the assortment. The gains are not large, which makes sense given that the early- and late-customer MNLs are not dramatically different (most coefficients differ by less than 0.1). The mixture solution is a reasonable compromise when the type is unknown.